# Create Issue Indices for Impresso-1 BNF and BNF-EN Titles

Generates issue-index JSON shards for the **11 legacy titles** already imported
in Impresso-1:

| Set | Batch tag | Titles | Has OLR |
|-----|-----------|--------|---------|
| 1 – Marché Presse | `BNF` | Excelsior, La-Fronde, Marie-Claire, Oeuvre | yes |
| 0 – Europeana | `BNF-EN` | legaulois, lematin, lepji, jdpl, oecaen, oerennes, lepetitparisien | yes |

Index schema (same as `issue_index.bnf.json`):
```json
{
  "day": "15", "edition": "a",
  "local_path": ["BNF/Excelsior/4600000"],
  "ark_id": "bpt6k46000007",
  "batch": "BNF"
}
```

**Ark strategy per set**
- *BNF*: extracted directly from `<fileIdentifier>` in the first ALTO file in `ocr/` — no API call.
- *BNF-EN*: fetched from the Gallica API via the existing `get_issues_iiif_arks` helper;
  the METS file contains no ark information.

## Imports and configuration

In [1]:
import gzip
import json
import logging
import os
import re
import string
import sys
import requests
from dotenv import load_dotenv
from collections import defaultdict
from datetime import datetime, date
from pathlib import Path

from bs4 import BeautifulSoup
from tqdm import tqdm
load_dotenv()

REPO_ROOT = Path(__file__).resolve().parents[3] if "__file__" in dir() else Path.cwd().parents[2]
sys.path.insert(0, str(REPO_ROOT))

# BNF helpers used only for date parsing (manifest.xml)
from text_preparation.importers.bnf.helpers import get_journal_name, parse_date
from text_preparation.importers.bnf.detect import DATE_FORMATS, DATE_SEPARATORS
from text_preparation.importers.mets_alto.mets import get_dmd_sec

# BNF-EN: API helper to fetch issue-level ark IDs per journal
from text_preparation.importers.bnf_en.detect import API_MAPPING, get_issues_iiif_arks

logging.basicConfig(level=logging.WARNING)

ORIGINAL_BASE = "/mnt/project_impresso/original"
INDEX_OUT_DIR = REPO_ROOT / "text_preparation/data/issue_indices"
INDEX_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root  :", REPO_ROOT)
print("Output dir :", INDEX_OUT_DIR)

Repo root  : /home/piconti/impresso-text-acquisition
Output dir : /home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices


## Part 1 – BNF (Marché Presse) — 4 legacy titles

On-disk layout:
```
BNF/<JournalDir>/<numeric_id>/
    manifest.xml   ← date + secondary_date live here
    ocr/
        X0000001.xml.gz   ← fileIdentifier holds the issue-level ark
    toc/
```

The issue-level ark is at:
`<fileIdentifier>ark:/12148/bpt6k46000007/f1</fileIdentifier>`
→ `ark_id = bpt6k46000007`

Same-day multi-edition issues are sorted by numeric folder id and assigned
edition letters `a`, `b`, `c`… in that order.

In [3]:
BNF_BASE_DIR = os.path.join(ORIGINAL_BASE, "BNF")
BNF_OUT_FILE = INDEX_OUT_DIR / "issue_index.bnf_mp.json"

LEGACY_BNF_DIRS = {
    "Excelsior":    "excelsior",
    "La-Fronde":    "lafronde",
    "Marie-Claire": "marieclaire",
    "Oeuvre":       "oeuvre",
}

In [3]:
def parse_bnf_manifest(manifest_path: str) -> tuple[date, date | None] | None:
    """Extract (date, secondary_date) from a BNF manifest.xml.

    Returns None when the file is missing or unparseable.
    The date is in dmdSec[@ID='DMD.2'] → mods:originInfo → mods:dateIssued.
    Delegates to the existing parse_date helper which handles the two BNF date
    formats (YYYY-MM-DD and YYYY/MM/DD) and dual-date issues.
    """
    try:
        with open(manifest_path, encoding="utf-8") as f:
            soup = BeautifulSoup(f, "xml")
        dmd2 = get_dmd_sec(soup, "DMD.2")
        if dmd2 is None:
            logging.warning("No DMD.2 in %s", manifest_path)
            return None
        raw_date = dmd2.find("date").contents[0]
        return parse_date(raw_date, DATE_FORMATS, DATE_SEPARATORS)
    except Exception as exc:
        logging.warning("Could not parse manifest %s: %s", manifest_path, exc)
        return None


def extract_issue_ark(ocr_dir: str) -> str | None:
    """Return the bare issue-level ark from the first ALTO file in ocr_dir.

    Format in file: <fileIdentifier>ark:/12148/bpt6k46000007/f1</fileIdentifier>
    Returns: 'bpt6k46000007'
    """
    try:
        alto_files = sorted(f for f in os.listdir(ocr_dir) if ".xml" in f)
        if not alto_files:
            logging.warning("No ALTO files in %s", ocr_dir)
            return None
        alto_path = os.path.join(ocr_dir, alto_files[0])
        opener = gzip.open if alto_path.endswith(".gz") else open
        with opener(alto_path) as fh:
            soup = BeautifulSoup(fh.read(), "xml")
        tag = soup.find("fileIdentifier")
        if tag is None:
            return None
        m = re.search(r"ark:/12148/([^/\s]+)", tag.get_text(strip=True))
        return m.group(1) if m else None
    except Exception as exc:
        logging.warning("ark extraction failed for %s: %s", ocr_dir, exc)
        return None


# Smoke-tests
_ark = extract_issue_ark("/mnt/project_impresso/original/BNF/Excelsior/4600000/ocr")
print("ark smoke-test :", _ark)          # → bpt6k46000007

_dt = parse_bnf_manifest("/mnt/project_impresso/original/BNF/Excelsior/4600000/manifest.xml")
print("date smoke-test:", _dt)           # → (date(1910, 11, 16), None)

ark smoke-test : bpt6k46000007
date smoke-test: (datetime.date(1910, 11, 16), None)


In [ ]:
def build_bnf_index(bnf_base: str, legacy_dirs: dict[str, str]) -> dict:
    """Build alias→year→month→[entries] for the 4 BNF Marché Presse legacy titles."""

    # Step 1 — collect raw records
    raw: list[dict] = []
    for dirname, alias in legacy_dirs.items():
        journal_path = os.path.join(bnf_base, dirname)
        if not os.path.isdir(journal_path):
            print(f"WARNING: {journal_path} not found — skipping")
            continue
        numeric_ids = sorted(d for d in os.listdir(journal_path) if d.isdigit())
        for num_id in tqdm(numeric_ids, desc=alias, leave=False):
            issue_path = os.path.join(journal_path, num_id)
            manifest   = os.path.join(issue_path, "manifest.xml")
            result     = parse_bnf_manifest(manifest)
            if result is None:
                continue
            np_date, secondary_date = result
            raw.append({
                "alias":          alias,
                "date":           np_date,
                "secondary_date": secondary_date,
                "path":           issue_path,
                "num_id":         num_id,
            })

    # Step 2 — assign edition letters per (alias, date), ordered by numeric id
    by_day: dict[tuple, list[dict]] = defaultdict(list)
    for r in raw:
        by_day[(r["alias"], r["date"])].append(r)
    for group in by_day.values():
        group.sort(key=lambda x: x["num_id"])
        for idx, r in enumerate(group):
            r["edition"] = string.ascii_lowercase[idx]

    # Step 3 — build index entries (extract ark from ALTO while we are here)
    index: dict = {}
    for r in tqdm(raw, desc="Building BNF entries"):
        alias = r["alias"]
        year  = str(r["date"].year)
        month = f"{r['date'].month:02d}"

        ocr_dir = os.path.join(r["path"], "ocr")
        ark_id  = extract_issue_ark(ocr_dir) if os.path.isdir(ocr_dir) else None
        rel     = os.path.relpath(r["path"], ORIGINAL_BASE)

        entry: dict = {
            "day":        f"{r['date'].day:02d}",
            "edition":    r["edition"],
            "local_path": [rel],
            "ark_id":     ark_id,
            "batch":      "BNF",
        }
        if r["secondary_date"] is not None:
            entry["secondary_date"] = str(r["secondary_date"])

        index.setdefault(alias, {}).setdefault(year, {}).setdefault(month, []).append(entry)

    # Step 4 — sort within each month
    for alias in index:
        for year in index[alias]:
            for month in index[alias][year]:
                index[alias][year][month].sort(key=lambda e: (e["day"], e["edition"]))

    return index


bnf_index = build_bnf_index(BNF_BASE_DIR, LEGACY_BNF_DIRS)

for alias, years in bnf_index.items():
    total = sum(len(es) for m in years.values() for es in m.values())
    print(f"  {alias}: {len(years)} years, {total} issues")

In [5]:
with open(BNF_OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(bnf_index, f, indent=2, ensure_ascii=False)
print(f"Saved → {BNF_OUT_FILE}")

alias = next(iter(bnf_index))
year  = next(iter(bnf_index[alias]))
month = next(iter(bnf_index[alias][year]))
print("\nSample entry:")
print(json.dumps(bnf_index[alias][year][month][0], indent=2))

Saved → /home/piconti/impresso-text-acquisition/text_preparation/data/issue_indices/issue_index.bnf_1.json

Sample entry:
{
  "day": "16",
  "edition": "a",
  "local_path": [
    "BNF/Excelsior/4600000"
  ],
  "ark_id": "bpt6k46000007",
  "batch": "BNF"
}


## Part 2 – BNF-EN (Europeana) — 7 legacy titles

On-disk layout:
```
BNF-EN/old/<JournalDir>/<YYYYMMDD>_<edition_int>/
    <YYYYMMDD>_<ed>-METS.xml
    ALTO/
        <YYYYMMDD>_<ed>-0001.xml  …
```

Date and edition come directly from the directory name.
The METS file contains **no ark information**.
Issue-level arks are fetched once from the Gallica API using `get_issues_iiif_arks`;
the function returns `[(canonical_issue_id, ark_id)]` pairs for a given journal.

#### Fetch the data

In [8]:
BNFEN_BASE_DIR = os.path.join(ORIGINAL_BASE, "BNF-EN", "old")
BNFEN_OUT_FILE = INDEX_OUT_DIR / "issue_index.bnf_en1.json"

EDITION_NUM_TO_LETTER = {i: string.ascii_lowercase[i - 1] for i in range(1, 10)}

def dirname_to_alias(dirname: str) -> str:
    """Mirror the normalisation in bnf_en/detect.py dir2issue."""
    return dirname.lower().replace("-", "").strip()

# Sanity-check
EXPECTED = set(API_MAPPING.keys())
found = {
    dirname_to_alias(d)
    for d in os.listdir(BNFEN_BASE_DIR)
    if os.path.isdir(os.path.join(BNFEN_BASE_DIR, d)) and not d.startswith(".")
}
print("Found  :", found)
print("Missing:", EXPECTED - found)
print("Extra  :", found - EXPECTED)

Found  : {'lepetitparisien', 'jdpl', 'oerennes', 'legaulois', 'lematin', 'oecaen', 'lepji'}
Missing: set()
Extra  : set()


In [9]:
API_MAPPING

{'oerennes': 'cb32830550k',
 'oecaen': 'cb41193642z',
 'lematin': 'cb328123058',
 'lepji': 'cb32836564q',
 'jdpl': 'cb39294634r',
 'legaulois': 'cb32779904b',
 'lepetitparisien': 'cb34419111x'}

In [ ]:
ark_map: dict[str, str] = {}

#def fetch_bnfen_arks(ark_map) -> dict[str, str]:
"""Fetch issue-level ark IDs for all 7 BNF-EN titles from the Gallica API.

Uses get_issues_iiif_arks (bnf_en/detect.py), which calls the Gallica
/services/Issues endpoint for each year.  Expect ~1-2 minutes total.

Returns:
    dict mapping canonical_issue_id → bare_ark_id
    e.g. {'legaulois-1884-04-08-a': 'bpt6k532073c', ...}
"""
for alias, journal_ark in tqdm(API_MAPPING.items(), desc="Fetching arks via Gallica API"):
    pairs = get_issues_iiif_arks((alias, journal_ark))
    for canonical_id, ark in pairs:
        ark_map[canonical_id] = ark
print(f"Fetched {len(ark_map)} issue arks across {len(API_MAPPING)} titles")
    #return ark_map

#bnfen_arks = fetch_bnfen_arks(ark_map)

In [ ]:
import pandas as pd
arks_list_filepath = "/mnt/project_impresso/original/BNF-EN/liste-arks.xls"

arks_list_df = pd.read_excel(arks_list_filepath, engine='xlrd')  # .xls needs xlrd
# for .xlsx files, use engine='openpyxl' (or omit engine, pandas auto-detects)
arks_list_df.head()

,CHEMIN_COMPLET,NB_PAGES,ARK_DOCNUM,ID_PERENNE_PROVENANCE,TITRE,ID_OUVRAGE
0,ROOT_ID\BnF_00014\1901\19011128,4,ark:/12148/bpt6k639441d,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639441
1,ROOT_ID\BnF_00014\1901\19011127,4,ark:/12148/bpt6k6394401,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639440
2,ROOT_ID\BnF_00014\1901\19011126,4,ark:/12148/bpt6k6394393,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639439
3,ROOT_ID\BnF_00014\1901\19011125,4,ark:/12148/bpt6k639438q,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639438
4,ROOT_ID\BnF_00014\1901\19011124,4,ark:/12148/bpt6k639437b,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639437


In [5]:
arks_list_df

,CHEMIN_COMPLET,NB_PAGES,ARK_DOCNUM,ID_PERENNE_PROVENANCE,TITRE,ID_OUVRAGE
0,ROOT_ID\BnF_00014\1901\19011128,4,ark:/12148/bpt6k639441d,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639441
1,ROOT_ID\BnF_00014\1901\19011127,4,ark:/12148/bpt6k6394401,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639440
2,ROOT_ID\BnF_00014\1901\19011126,4,ark:/12148/bpt6k6394393,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639439
3,ROOT_ID\BnF_00014\1901\19011125,4,ark:/12148/bpt6k639438q,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639438
4,ROOT_ID\BnF_00014\1901\19011124,4,ark:/12148/bpt6k639437b,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-639437
...,...,...,...,...,...,...
15722,ROOT_ID\BnF_00014\1942\19420115,4,ark:/12148/bpt6k662547s,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-662547
15723,ROOT_ID\BnF_00014\1942\19420114,4,ark:/12148/bpt6k662546d,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-662546
15724,ROOT_ID\BnF_00014\1942\19420113,2,ark:/12148/bpt6k6625451,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-662545
15725,ROOT_ID\BnF_00014\1942\19420112,4,ark:/12148/bpt6k662544n,ark:/12148/cb32830550k,L'Ouest-Éclair : journal quotidien d'informati...,NUMM-662544


In [4]:
arks_list_df.groupby('TITRE').count()

,CHEMIN_COMPLET,NB_PAGES,ARK_DOCNUM,ID_PERENNE_PROVENANCE,ID_OUVRAGE
TITRE,,,,,
"L'Ouest-Éclair : journal quotidien d'informations, politique, littéraire, commercial",15727,15727,15727,15727,15727


A priori we're having problems to fetch this specific info using the previous API. 

I Have not found the new versions of these requests yet, so another option might be to fetch the ARK id and relevant information from the already generated canonical data which currently lives on S3.

In [ ]:
def parse_bnfen_dirname(dirname: str) -> tuple[date, str] | None:
    """Parse '{YYYYMMDD}_{edition_int}' → (date, edition_letter) or None."""
    m = re.fullmatch(r"(\d{8})_(\d+)", dirname)
    if not m:
        return None
    try:
        d = datetime.strptime(m.group(1), "%Y%m%d").date()
        edition = EDITION_NUM_TO_LETTER.get(int(m.group(2)),
                                            string.ascii_lowercase[int(m.group(2)) - 1])
        return d, edition
    except (ValueError, IndexError) as exc:
        logging.warning("Cannot parse dirname %s: %s", dirname, exc)
        return None


def build_bnfen_index(base_dir: str, arks: dict[str, str]) -> dict:
    """Build alias→year→month→[entries] for the 7 BNF-EN Europeana legacy titles."""
    index: dict = {}
    missing_arks: list[str] = []

    for journal_dirname in sorted(
        d for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d)) and not d.startswith(".")
    ):
        alias        = dirname_to_alias(journal_dirname)
        journal_path = os.path.join(base_dir, journal_dirname)

        for issue_dirname in tqdm(sorted(os.listdir(journal_path)), desc=alias, leave=False):
            issue_path = os.path.join(journal_path, issue_dirname)
            if not os.path.isdir(issue_path):
                continue
            parsed = parse_bnfen_dirname(issue_dirname)
            if parsed is None:
                logging.warning("Skipping unrecognised dir: %s", issue_path)
                continue
            issue_date, edition = parsed

            canonical_id = f"{alias}-{issue_date.year}-{issue_date.month:02d}-{issue_date.day:02d}-{edition}"
            ark_id = arks.get(canonical_id)
            if ark_id is None:
                missing_arks.append(canonical_id)

            entry = {
                "day":        f"{issue_date.day:02d}",
                "edition":    edition,
                "local_path": [os.path.relpath(issue_path, ORIGINAL_BASE)],
                "ark_id":     ark_id,
                "batch":      "BNF-EN",
            }
            year  = str(issue_date.year)
            month = f"{issue_date.month:02d}"
            index.setdefault(alias, {}).setdefault(year, {}).setdefault(month, []).append(entry)

    for alias in index:
        for year in index[alias]:
            for month in index[alias][year]:
                index[alias][year][month].sort(key=lambda e: (e["day"], e["edition"]))

    if missing_arks:
        print(f"WARNING: {len(missing_arks)} issues had no API ark — ark_id will be null")
        for cid in missing_arks[:10]:
            print(" ", cid)
        if len(missing_arks) > 10:
            print(f"  ... and {len(missing_arks) - 10} more")

    return index


bnfen_index = build_bnfen_index(BNFEN_BASE_DIR, bnfen_arks)

for alias, years in bnfen_index.items():
    total = sum(len(es) for m in years.values() for es in m.values())
    print(f"  {alias}: {len(years)} years, {total} issues")

In [ ]:
with open(BNFEN_OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(bnfen_index, f, indent=2, ensure_ascii=False)
print(f"Saved → {BNFEN_OUT_FILE}")

alias = next(iter(bnfen_index))
year  = next(iter(bnfen_index[alias]))
month = next(iter(bnfen_index[alias][year]))
print("\nSample entry:")
print(json.dumps(bnfen_index[alias][year][month][0], indent=2))

## Validation

In [ ]:
def count_index(index: dict) -> dict:
    issues, missing_arks = 0, 0
    years: set = set()
    for alias, yd in index.items():
        years |= set(yd)
        for year, md_ in yd.items():
            for month, entries in md_.items():
                issues += len(entries)
                missing_arks += sum(1 for e in entries if e.get("ark_id") is None)
    return {"aliases": len(index), "issues": issues,
            "years": sorted(years), "missing_arks": missing_arks}


def check_duplicates(index: dict, label: str) -> None:
    seen: set = set()
    dupes: list = []
    for alias, yd in index.items():
        for year, md_ in yd.items():
            for month, entries in md_.items():
                for e in entries:
                    key = (alias, year, month, e["day"], e["edition"])
                    if key in seen:
                        dupes.append(key)
                    seen.add(key)
    if dupes:
        print(f"[{label}] WARNING: {len(dupes)} duplicate keys!")
        for d in dupes[:5]:
            print(" ", d)
    else:
        print(f"[{label}] No duplicates — OK")


for label, idx in [("BNF", bnf_index), ("BNF-EN", bnfen_index)]:
    c = count_index(idx)
    print(f"=== {label} ===")
    print(f"  Aliases      : {c['aliases']}")
    print(f"  Issues       : {c['issues']}")
    print(f"  Year range   : {c['years'][0]} – {c['years'][-1]}")
    print(f"  Missing arks : {c['missing_arks']}")
    check_duplicates(idx, label)
    print()
print("Output files:")
print(f"  {BNF_OUT_FILE}")
print(f"  {BNFEN_OUT_FILE}")

### Mapping each alias to its BNF data format

There are 3 different formats of BNF data:
- **MP**: Marché Presse titles, with OLR - old and new data
- **EN**: Europeana titles, with OLR - old data (new does not have olr)
- **BNF**: All no-OLR titles, all new data to import.

We want to have one issue_index file per format.
We already have:
- `impresso-text-acquisition/text_preparation/data/issue_indices/issue_index.bnf_new.json` which lists all the issues in the new BNF data, and needs to be dispatched to one of the three formats above.
- `impresso-text-acquisition/text_preparation/data/issue_indices/issue_index.bnf_1.json` which lists all the legacy issues in the MP format.


In [2]:
alias_to_format = {
  "abendland": "MP-OLR",
  "actionfrancaise1899": "BNF-OCR",
  "actionfrancaise1908": "BNF-OCR",
  "alsacefrancaise": "MP-OLR",
  "annalescol": "BNF-OCR",
  "armeecoloniale": "BNF-OCR",
  "aurore1943": "MP-OLR",
  "bcaf": "BNF-OCR",
  "bocfo": "MP-OLR",
  "bonnetrouge": "MP-OLR",
  "bouffon": "MP-OLR",
  "bsecm": "BNF-OCR",
  "bsgecm": "BNF-OCR",
  "bulletincol": "BNF-OCR",
  "bulletinoffcol": "BNF-OCR",
  "canardenchaine": "BNF-OCR",
  "candide": "MP-OLR",
  "cesoir": "MP-OLR",
  "chicagotribune": "MP-OLR",
  "cinejournal": "MP-OLR",
  "combat": "MP-OLR",
  "courriercolill": "BNF-OCR",
  "courrierlondres": "MP-OLR",
  "courriermarna": "BNF-OCR",
  "cridespeuples": "BNF-OCR",
  "cripeuple1871": "MP-OLR",
  "defensenationalparis": "MP-OLR",
  "democratiepacifique": "MP-OLR",
  "demokratischeztg": "MP-OLR",
  "depechecolill": "BNF-OCR",
  "depechetoulouse": "MP-OLR",
  "echangouleme": "MP-OLR",
  "echoalger": "MP-OLR",
  "echoran": "MP-OLR",
  "elouma": "BNF-OCR",
  "etatsuniseurope": "MP-OLR",
  "europeartistique": "MP-OLR",
  "europecoloniale": "MP-OLR",
  "europecolonies": "MP-OLR",
  "europedabord": "MP-OLR",
  "europedanubienne": "MP-OLR",
  "europefinanciere1": "MP-OLR",
  "europefinanciere2": "MP-OLR",
  "europefuture": "MP-OLR",
  "europeill": "MP-OLR",
  "europeillrev": "MP-OLR",
  "europeindcom": "MP-OLR",
  "europejq": "MP-OLR",
  "europenouvelle": "MP-OLR",
  "europeor1919": "MP-OLR",
  "europeorroum": "MP-OLR",
  "europepscil": "MP-OLR",
  "excelsior": "MP-OLR",
  "figaro1826": "BNF-OCR",
  "figaro1839": "MP-OLR",
  "figaro1854": "BNF-OCR",
  "figarosupl": "BNF-OCR",
  "franceboheme": "MP-OLR",
  "franceeuropeor": "MP-OLR",
  "francelibre": "MP-OLR",
  "francerussie": "MP-OLR",
  "francesoir": "MP-OLR",
  "franceyougoslavie": "MP-OLR",
  "franctireur": "MP-OLR",
  "freeeurope": "MP-OLR",
  "gazettecoloniale": "BNF-OCR",
  "grandechonord": "MP-OLR",
  "gringoire": "MP-OLR",
  "humanite": "BNF-OCR",
  "ikdam": "BNF-OCR",
  "intransigeant": "BNF-OCR",
  "jdpl": "EN-OLR",
  "jeuneeurope1930": "MP-OLR",
  "jeuneeuroperev": "MP-OLR",
  "lacharente": "MP-OLR",
  "lacroix": "MP-OLR",
  "lafranceparis": "MP-OLR",
  "lafrique1844": "MP-OLR",
  "lafronde": "MP-OLR",
  "lagrimace": "BNF-OCR",
  "lajustice": "BNF-OCR",
  "laliberte": "MP-OLR",
  "lapresse": "BNF-OCR",
  "lauto": "MP-OLR",
  "lecaucase": "MP-OLR",
  "leconstitutionnel": "BNF-OCR",
  "lecorsaire": "MP-OLR",
  "lecourrier": "MP-OLR",
  "legaulois": "EN-OLR",
  "lejournal": "MP-OLR",
  "lematin": "EN-OLR",
  "lepays": "MP-OLR",
  "lepetitjournal": "BNF-OCR",
  "lepetitparisien": "EN-OLR",
  "lepeuple": "MP-OLR",
  "lepji": "EN-OLR",
  "lesbalkans": "MP-OLR",
  "lescolonies": "BNF-OCR",
  "letemps": "BNF-OCR",
  "lettresfrancaises": "MP-OLR",
  "liberation": "MP-OLR",
  "libertecol": "BNF-OCR",
  "marieclaire": "MP-OLR",
  "mondecolill": "BNF-OCR",
  "notretemps": "MP-OLR",
  "nouvellefrancemars": "BNF-OCR",
  "nouvellestcheco": "MP-OLR",
  "nyherald": "MP-OLR",
  "oecaen": "EN-OLR",
  "oenantes": "EN-OLR",
  "oerennes": "EN-OLR",
  "oeuvre": "MP-OLR",
  "paixtravail": "MP-OLR",
  "parisbalkans": "MP-OLR",
  "pariseurope": "MP-OLR",
  "parismidi": "MP-OLR",
  "paristribune": "MP-OLR",
  "petitepresse": "MP-OLR",
  "petitmarocain": "MP-OLR",
  "pressecolill": "BNF-OCR",
  "progrescol": "BNF-OCR",
  "quinzainecol": "BNF-OCR",
  "renaissancecol": "BNF-OCR",
  "revuecol": "BNF-OCR",
  "revuecolan": "BNF-OCR",
  "revueorienth": "MP-OLR",
  "rhcf": "BNF-OCR",
  "rqcm": "BNF-OCR",
  "tablettescol": "BNF-OCR",
  "terreeurope": "MP-OLR",
  "togocameroun": "BNF-OCR",
  "unionfrancaise": "MP-OLR",
  "univers": "BNF-OCR",
  "vendredi": "MP-OLR",
  "vielatine": "MP-OLR",
  "vsve": "MP-OLR"
}

#### Split the existing issue_index.bnf_new.json into the three formats

First read them in, then map them to the correct format based on the title alias.
Finally, re-write the issue_index files for each format.

In [4]:
# input files
new_data_index = os.path.join(INDEX_OUT_DIR, "issue_index.bnf_new.json")
bnf_1_data_index = os.path.join(INDEX_OUT_DIR, "issue_index.bnf_1.json")

# output files
bnf_mp_data_index = os.path.join(INDEX_OUT_DIR, "issue_index.bnf_mp.json")
bnf_en_data_index = os.path.join(INDEX_OUT_DIR, "issue_index.bnf_en.json")
bnf_ocr_data_index = os.path.join(INDEX_OUT_DIR, "issue_index.bnf_ocr.json")

In [5]:
with open(new_data_index, 'r', encoding='utf-8') as fin:
    new_issue_dict = json.load(fin)

with open(bnf_1_data_index, 'r', encoding='utf-8') as fin:
    bnf_1_issue_dict = json.load(fin)

In [ ]:
print(len(bnf_1_issue_dict), bnf_1_issue_dict.keys())
len(new_issue_dict), new_issue_dict.keys()

4 dict_keys(['excelsior', 'lafronde', 'marieclaire', 'oeuvre'])


(130,
 dict_keys(['jeuneeurope1930', 'letemps', 'quinzainecol', 'vsve', 'petitepresse', 'petitmarocain', 'bsgecm', 'lescolonies', 'pressecolill', 'lafrique1844', 'liberation', 'lecaucase', 'unionfrancaise', 'jeuneeuroperev', 'combat', 'franceyougoslavie', 'ikdam', 'democratiepacifique', 'terreeurope', 'europeor1919', 'gazettecoloniale', 'bocfo', 'nyherald', 'cridespeuples', 'etatsuniseurope', 'europedabord', 'europecoloniale', 'lepays', 'libertecol', 'lesbalkans', 'parisbalkans', 'demokratischeztg', 'depechetoulouse', 'progrescol', 'bulletinoffcol', 'revuecolan', 'oeuvre', 'europedanubienne', '.manifest_cache', 'laliberte', 'figarosupl', 'armeecoloniale', 'lacharente', 'elouma', 'chicagotribune', 'europeindcom', 'europefuture', 'nouvellestcheco', 'parismidi', 'europeartistique', 'lepeuple', 'notretemps', 'europefinanciere1', 'candide', 'francerussie', 'courrierlondres', 'bulletincol', 'mondecolill', 'lacroix', 'intransigeant', 'lecourrier', 'lapresse', 'vendredi', 'europenouvelle', 'eu

In [9]:
mp_issue_dict = {}
en_issue_dict = {}
ocr_issue_dict = {}
full_issue_dict = {
    "BNF-OCR": ocr_issue_dict,
    "MP-OLR": mp_issue_dict,
    "EN-OLR": en_issue_dict
}
all_index_files = {
    "BNF-OCR": bnf_ocr_data_index,
    "MP-OLR": bnf_mp_data_index,
    "EN-OLR": bnf_en_data_index
}

for alias, format in alias_to_format.items():
    if alias in new_issue_dict:
        full_issue_dict[format][alias] = new_issue_dict[alias]
    elif alias in bnf_1_issue_dict:
        full_issue_dict[format][alias] = bnf_1_issue_dict[alias]
    else:
        print(f"Alias {alias} not in any issue index yet, still to be created")

for format, issue_dict in full_issue_dict.items():
    if len(issue_dict) != 0:
        print(f"Format {format} - writing the index for {len(issue_dict)} aliases to disk.")
        with open(all_index_files[format], 'w', encoding='utf-8') as fout:
            json.dump(issue_dict, fout, indent=2)
    else:
        print(f"Issue dict for format {format} is empty, not writing it yet")

Alias jdpl not in any issue index yet, still to be created
Alias legaulois not in any issue index yet, still to be created
Alias lematin not in any issue index yet, still to be created
Alias lepetitparisien not in any issue index yet, still to be created
Alias lepji not in any issue index yet, still to be created
Alias oecaen not in any issue index yet, still to be created
Alias oerennes not in any issue index yet, still to be created
Format BNF-OCR - writing the index for 43 aliases to disk.


Format MP-OLR - writing the index for 85 aliases to disk.
Format EN-OLR - writing the index for 1 aliases to disk.
